In [ ]:
import pandas as pd
import numpy as np
from scipy.sparse import save_npz
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from tensorflow.keras.preprocessing.text import Tokenizer
from  tensorflow.keras.preprocessing.sequenceimport pad_sequences
from pathlib import Path
import joblib
import pickle

In [29]:
processed_data_dir = Path("../data/processed")
models_dir = Path("../models")

processed_data_dir.mkdir(parents=True, exist_ok=True)
models_dir.mkdir(parents=True, exist_ok=True)

In [30]:
raw_data_path = Path("../data/raw/phishing_email.csv")
df = pd.read_csv(raw_data_path)
df.head()

,text_combined,label
0,hpl nom may 25 2001 see attached file hplno 52...,0
1,nom actual vols 24 th forwarded sabrae zajac h...,0
2,enron actuals march 30 april 1 201 estimated a...,0
3,hpl nom may 30 2001 see attached file hplno 53...,0
4,hpl nom june 1 2001 see attached file hplno 60...,0


In [31]:
df.shape

(82486, 2)

In [32]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 82486 entries, 0 to 82485
Data columns (total 2 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   text_combined  82486 non-null  str  
 1   label          82486 non-null  int64
dtypes: int64(1), str(1)
memory usage: 1.3 MB


In [33]:
df.columns = ['text', 'label']
df.head()

,text,label
0,hpl nom may 25 2001 see attached file hplno 52...,0
1,nom actual vols 24 th forwarded sabrae zajac h...,0
2,enron actuals march 30 april 1 201 estimated a...,0
3,hpl nom may 30 2001 see attached file hplno 53...,0
4,hpl nom june 1 2001 see attached file hplno 60...,0


In [34]:
df['label'].value_counts()

label
1    42891
0    39595
Name: count, dtype: int64

In [35]:
X_train, X_, y_train, y_ = train_test_split(
    df['text'], df['label'], test_size=0.2, random_state=42
)
X_test, X_val, y_test, y_val = train_test_split(
    X_, y_, test_size=0.5, random_state=42
)

#### Logistic Regression Input Prepration

In [36]:
tfidf_vectorizer = TfidfVectorizer(
    lowercase=True,
    stop_words='english',
    max_features=30000,
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.95
)

In [37]:
X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)
X_val_tfidf = tfidf_vectorizer.transform(X_val)
X_test_tfidf = tfidf_vectorizer.transform(X_test)

In [38]:
X_train_tfidf.shape, X_val_tfidf.shape, X_test_tfidf.shape

((65988, 30000), (8249, 30000), (8249, 30000))

In [39]:
joblib.dump(tfidf_vectorizer, models_dir / "tfidf_vectorizer.joblib")

save_npz(processed_data_dir / "X_train_tfidf.npz", X_train_tfidf)
save_npz(processed_data_dir / "X_val_tfidf.npz", X_val_tfidf)
save_npz(processed_data_dir / "X_test_tfidf.npz", X_test_tfidf)
y_train.to_csv(processed_data_dir / "y_train.csv", index=False, header=['label'])
y_val.to_csv(processed_data_dir / "y_val.csv", index=False, header=['label'])
y_test.to_csv(processed_data_dir / "y_test.csv", index=False, header=['label'])

#### FNN-RNN-LSTM Input Prepration

In [45]:
tockenizer = Tokenizer(
    num_words=30000, 
    oov_token="<OOV>",
    lower=True
)
tockenizer.fit_on_texts(X_train)

In [46]:
X_train_seq = tockenizer.texts_to_sequences(X_train)
X_val_seq = tockenizer.texts_to_sequences(X_val)
X_test_seq = tockenizer.texts_to_sequences(X_test)

In [47]:
X_train_seq = pad_sequences(X_train_seq, maxlen=500, padding='post', truncating='post')
X_val_seq = pad_sequences(X_val_seq, maxlen=500, padding='post', truncating='post')
X_test_seq = pad_sequences(X_test_seq, maxlen=500, padding='post', truncating='post')

In [48]:
pickle.dump(tockenizer, open(models_dir / "tokenizer.pkl", "wb"))

np.save(processed_data_dir / "X_train_seq.npy", X_train_seq)
np.save(processed_data_dir / "X_val_seq.npy", X_val_seq)
np.save(processed_data_dir / "X_test_seq.npy", X_test_seq)